## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-25-13-53-24 +0000,nyt,Another Hot Inflation Reading Keeps Fed on Tra...,https://www.nytimes.com/2026/11/25/business/ec...
1,2026-11-25-13-52-00 +0000,wsj,October Durable Goods Orders Rose Less Than Ex...,https://www.wsj.com/economy/october-durable-go...
2,2026-11-25-13-51-43 +0000,nyt,Trump Administration Unveils New Tariffs for C...,https://www.nytimes.com/2026/11/25/business/tr...
3,2026-11-25-13-49-47 +0000,bbc,US announces new tariffs on host of countries ...,https://www.bbc.co.uk/news/articles/c893r9ke5p...
4,2026-11-25-13-48-36 +0000,nyt,How an Iranian Ally Gained the Upper Hand in Y...,https://www.nytimes.com/2026/11/25/world/middl...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2715/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
15,trump,38
235,nvidia,35
415,huang,21
20,china,21
136,ai,19


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
65,2026-11-25-11-58-00 +0000,wsj,Nvidia’s Jensen Huang is set for at least two ...,https://www.wsj.com/politics/policy/trump-says...,147
220,2026-11-25-00-18-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...,147
356,2026-11-24-18-28-12 +0000,nypost,Trump says cheating Nvidia CEO Jensen Huang an...,https://nypost.com/2026/11/24/us-news/trump-sa...,136
206,2026-11-25-01-12-13 +0000,nypost,"Marco Rubio’s concern about Jensen Huang, Nvid...",https://nypost.com/2026/11/24/us-news/rubio-je...,125
121,2026-11-25-10-00-00 +0000,wapo,"In backing Nvidia’s Huang, Trump deals his Chi...",https://www.washingtonpost.com/politics/2026/1...,121


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
65,147,2026-11-25-11-58-00 +0000,wsj,Nvidia’s Jensen Huang is set for at least two ...,https://www.wsj.com/politics/policy/trump-says...
220,72,2026-11-25-00-18-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...
34,63,2026-11-25-12-59-03 +0000,startribune,‘No risk’ to Minnesota as Army set to blow up ...,https://www.startribune.com/iowa-lead-detonati...
128,63,2026-11-25-09-56-46 +0000,wsj,Stock Market Today: Tech-Led Slide Deepens as ...,https://www.wsj.com/livecoverage/stock-market-...
98,54,2026-11-25-11-00-00 +0000,nypost,New bill would allow prediction markets like K...,https://nypost.com/2026/11/25/us-news/new-bill...
227,48,2026-11-24-23-49-03 +0000,nypost,Hudson River Trading expanding at 3 World Tra...,https://nypost.com/2026/11/24/business/hudson-...
336,45,2026-11-24-20-06-02 +0000,latimes,A man's severed head was found in a rural Cali...,https://www.latimes.com/california/story/2026-...
195,45,2026-11-25-02-00-00 +0000,wsj,Pennsylvania Gov. Josh Shapiro is staking out ...,https://www.wsj.com/politics/policy/shapiro-ac...
221,40,2026-11-25-00-13-52 +0000,nypost,"Andrew Giuliani’s daughter, 4, steals spotligh...",https://nypost.com/2026/11/24/us-news/andrew-g...
39,39,2026-11-25-12-53-36 +0000,nypost,"Kilmar Ábrego García, who fought being mistake...",https://nypost.com/2026/11/25/world-news/kilma...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
